# Expert-CFG 베이스라인 재현 — PathVQA

논문: *Uncertainty-Driven Expert Control: Enhancing the Reliability of Medical Vision-Language Models* (arXiv:2507.09209)

이 노트북은 RAG / Expert-CFG 를 붙이기 **전 단계**인 **Phi3.5V-Med 4.2B 베이스라인**을 PathVQA 에서 재현합니다.

**재현 목표 (Table 1, Phi3.5V-Med 4.2B, PathVQA)**

| Open (recall) | Closed (acc) | Overall |
|---|---|---|
| 37.42 | 92.89 | 65.24 |

**논문에서 확인한 구성 요소**

| 항목 | 값 | 출처 |
|---|---|---|
| Base VLM | Phi-3.5-Vision (4.2B, CLIP ViT-L/14) | §3.1, §4.2 |
| Med 사전학습 | PubMedVision 2-stage (Alignment → Instruction), LoRA | §3.1 — 공개 LoRA `Phi-3.5V-Med` 로 대체 |
| LoRA | rank 64 (repo 기본: alpha = 2×rank = 128, dropout 0) | §4.3, `demo.py` |
| Downstream FT | AdamW, cosine, lr 1e-4, batch 64, 10 epochs | §4.3 |
| 정밀도 | bfloat16 | §4.3 |
| 디코딩 | greedy (num_beams=1), max_new_tokens 64 | §3.2, `demo.py` |
| 평가 | Closed: accuracy / Open: GT 토큰 recall (LLaVA-Med 방식) | §4.3 |
| Overall | 문항 수 가중 평균 (VQA-RAD 행으로 검산: (69.27·179+90.44·272)/451 = 82.04) | Table 1 |
| 불확실성 | 길이 정규화 predictive entropy, Eq.(4) | §3.2 |

**논문에 명시되지 않아 가정한 것** (결과 차이의 원인이 될 수 있음)
- downstream LoRA 방식: Med LoRA 를 merge 후 새 LoRA 학습 (기본) vs. Med LoRA 이어서 학습 → `DOWNSTREAM_MODE` 로 전환 가능
- LoRA target module: Phi-3 CookBook (이 repo 의 `demo.py` 원본) 정규식 — `qkv_proj, o_proj, gate_up_proj, down_proj`
- warmup, weight decay, num_crops, 비전 인코더 동결 여부

In [1]:
!git clone https://github.com/Minseongkim-dev/Expert-CFG.git /content/Expert-CFG

Cloning into '/content/Expert-CFG'...
remote: Enumerating objects: 59, done.
remote: Counting objects: 100% (4/4), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 59 (delta 2), reused 0 (delta 0), pack-reused 55 (from 1)
Receiving objects: 100% (59/59), 392.21 KiB | 65.37 MiB/s, done.
Resolving deltas: 100% (11/11), done.


## 0. 환경 확인

`requirements.txt` 는 `torch==2.3.0`, `transformers==4.43.0` 고정입니다. torch 2.3 은 Python 3.13 wheel 이 없으므로
README 대로 **Python 3.10 conda env** 를 커널로 쓰세요. 추가로 `peft` 가 필요합니다.

```
conda create -n expert_cfg python=3.10 -y && conda activate expert_cfg
pip install -r requirements.txt
pip install peft==0.11.1 datasets scikit-learn matplotlib ipykernel
```

In [2]:
from google.colab import drive
drive.mount('/content/drive')

OUTPUT_DIR  = '/content/drive/MyDrive/expert_cfg/logs_phi35_pathvqa_baseline'
RESULT_JSON = '/content/drive/MyDrive/expert_cfg/pathvqa_baseline_test.json'
EPOCHS = 5  # 10 → 5



Mounted at /content/drive


In [9]:
# peft 0.21+ 는 torchao>=0.16.0 을 요구합니다
import importlib, subprocess, sys

def _pkg_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None

_torchao_ver = _pkg_version('torchao')
if _torchao_ver is None or tuple(int(x) for x in _torchao_ver.split('.')[:2]) < (0, 16):
    print(f'torchao {_torchao_ver} → 업그레이드 중...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torchao>=0.16.0'])
    print('torchao 업그레이드 완료. 런타임을 재시작하지 않아도 됩니다.')
else:
    print(f'torchao {_torchao_ver} OK')

torchao 0.10.0 → 업그레이드 중...
torchao 업그레이드 완료. 런타임을 재시작하지 않아도 됩니다.


In [3]:
import os, sys, json, glob, random
from pathlib import Path

import numpy as np
import torch

def find_repo_root():
    candidates = []

    for start in [Path.cwd(), Path("/content"), Path("/workspace")]:
        if start.exists():
            candidates.extend([start, *start.iterdir()])

    for candidate in candidates:
        if candidate.is_dir() and (
            candidate / "src" / "highlighter_modules"
        ).is_dir():
            return candidate.resolve()

    # 현재 경로의 상위 디렉터리도 확인
    current = Path.cwd().resolve()
    for parent in [current, *current.parents]:
        if (parent / "src" / "highlighter_modules").is_dir():
            return parent

    raise FileNotFoundError(
        "repo root를 찾지 못했습니다. /content/Expert-CFG에 repo를 clone하거나 "
        "REPO_ROOT를 직접 지정하세요."
    )

REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import transformers
import peft

print("현재 작업 디렉터리:", Path.cwd())
print("repo         :", REPO_ROOT)
print("python       :", sys.version.split()[0])
print("torch        :", torch.__version__)
print("transformers :", transformers.__version__)
print("peft         :", peft.__version__)
print("CUDA         :", torch.cuda.is_available())

USE_BF16 = bool(
    torch.cuda.is_available() and torch.cuda.is_bf16_supported()
)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_DTYPE = torch.bfloat16 if USE_BF16 else torch.float32

if torch.cuda.is_available():
    print("GPU          :", torch.cuda.get_device_name(0))
    print("BF16         :", USE_BF16)

현재 작업 디렉터리: /content/Expert-CFG
repo         : /content/Expert-CFG
python       : 3.13.16
torch        : 2.11.0+cu130
transformers : 5.20.2
peft         : 0.21.2
CUDA         : True
GPU          : Tesla T4
BF16         : True


## 1. 설정 (논문 하이퍼파라미터)

In [4]:
# ── 경로 ───────────────────────────────────────────────────────────────
BASE_MODEL_DIR = './Phi-3.5-vision-instruct'          # HF 가중치를 이 폴더에 병합 (README)
MED_LORA_DIR   = './lora_weights/logs_phi35_pubmed_instruct'   # demo.py 의 Phi3.5V-Med LoRA 경로
MED_LORA_HF    = 'ecoxial2007/Phi-3.5V-Med'           # 로컬에 없으면 여기서 다운로드
PATHVQA_DIR    = './data/PathVQA'
OUTPUT_DIR     = './lora_weights/logs_phi35_pathvqa_baseline'
RESULT_JSON    = './examples/results/pathvqa_baseline_test.json'

# ── downstream LoRA 방식 (논문 미명시) ─────────────────────────────────
#   'merge_then_new' : Med LoRA 를 base 에 merge → 새 LoRA(rank 64) 학습
#   'continue'       : Med LoRA 를 trainable 로 로드해 그대로 이어서 학습
DOWNSTREAM_MODE = 'merge_then_new'

# ── LoRA (§4.3 rank 64 / demo.py: alpha ratio 2, dropout 0) ────────────
LORA_RANK        = 64
LORA_ALPHA_RATIO = 2
LORA_DROPOUT     = 0.0
# 신버전 PEFT(0.13+)는 regex 대신 list 사용
LORA_TARGET = ['qkv_proj', 'o_proj', 'gate_up_proj', 'down_proj']

# ── Downstream fine-tuning (§4.3) ──────────────────────────────────────
LR            = 1e-4
GLOBAL_BATCH  = 64
EPOCHS        = 5        # 비용 절약: 10 → 5 (논문은 10)
PER_DEVICE_BS = 1        # Phi-3-V collator 는 batch_size == 1 만 지원
WORLD_SIZE    = int(os.environ.get('WORLD_SIZE', 1))
GRAD_ACCUM    = GLOBAL_BATCH // (PER_DEVICE_BS * WORLD_SIZE)
WARMUP_RATIO  = 0.03     # 미명시 (CookBook 기본값 계열)
WEIGHT_DECAY  = 0.0      # 미명시
FREEZE_VISION = True     # 미명시 — demo.py 에 freeze 옵션 존재, 3090 24GB 메모리 고려

# ── 추론 (§3.2 greedy) ─────────────────────────────────────────────────
NUM_CROPS      = 16      # demo.py 기본값
MAX_NEW_TOKENS = 64
NUM_BEAMS      = 1

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.dirname(RESULT_JSON), exist_ok=True)
print(f'LoRA r={LORA_RANK}, alpha={LORA_RANK*LORA_ALPHA_RATIO}, grad_accum={GRAD_ACCUM} (global batch {GLOBAL_BATCH})')

LoRA r=64, alpha=128, grad_accum=64 (global batch 64)


## 2. PathVQA 데이터

원본 split: train 19,755 / val 6,279 / test 6,761 (총 32,795 ≈ 논문 32,799). closed = yes/no 답변.

두 가지 소스를 지원합니다.
1. **로컬 (repo 포맷, 권장)** — `src/datasets/pathvqa.py` 와 동일한 구조:
   `data/PathVQA/{split}.json` (각 항목 `img_id`, `question`, `answer`) + 이미지 `data/PathVQA/images/{split}/{img_id}.jpg`
2. **HuggingFace `flaviagiammarino/path-vqa`** — 로컬 파일이 없을 때 자동 사용.
   이 버전은 중복 (image, question, answer) 을 제거해 문항 수가 원본보다 약간 적으므로 수치가 소폭 달라질 수 있습니다.

In [5]:
from PIL import Image

def answer_type_of(ans):
    return 'CLOSED' if str(ans).strip().lower() in ('yes', 'no') else 'OPEN'

class PathVQASplit(torch.utils.data.Dataset):
    # 각 샘플: {'image': PIL, 'question', 'answer', 'answer_type'}
    def __init__(self, split):
        local_json = os.path.join(PATHVQA_DIR, f'{split}.json')
        if os.path.exists(local_json):
            with open(local_json) as f:
                self.records = json.load(f)
            self.source = 'local'
            self.img_root = os.path.join(PATHVQA_DIR, 'images')
        else:
            from datasets import load_dataset
            hf_split = {'val': 'validation'}.get(split, split)
            self.records = load_dataset('flaviagiammarino/path-vqa', split=hf_split,
                                        cache_dir=os.path.join(PATHVQA_DIR, 'hf_cache'))
            self.source = 'hf'

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        r = self.records[idx]
        if self.source == 'local':
            img_id = r['img_id']
            image = Image.open(os.path.join(self.img_root, img_id.split('_')[0], img_id + '.jpg'))
        else:
            image = r['image']
        answer = str(r['answer']).strip()
        return {'image': image.convert('RGB'), 'question': str(r['question']).strip(),
                'answer': answer, 'answer_type': answer_type_of(answer)}

train_ds = PathVQASplit('train')
test_ds  = PathVQASplit('test')

def type_counts(ds):
    answers = ([r['answer'] for r in ds.records] if ds.source == 'local' else ds.records['answer'])
    closed = sum(answer_type_of(a) == 'CLOSED' for a in answers)
    return closed, len(answers) - closed

for name, ds in [('train', train_ds), ('test', test_ds)]:
    c, o = type_counts(ds)
    print(f'{name:5s} [{ds.source}] total={len(ds):6d}  closed={c:5d}  open={o:5d}')

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/4.29k [00:00<?, ?B/s]

data/train-00000-of-00007-f2d0e9ef9f022d(…): reconstructing file:   0%|          |  0.00B / 42.8MB            

data/train-00000-of-00007-f2d0e9ef9f022d(…): downloading bytes:           |  0.00B            

data/train-00001-of-00007-47d8e0220bf6c9(…): reconstructing file:   0%|          |  0.00B / 81.0MB            

...

data/test-00002-of-00003-162830843501982(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/19654 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/6259 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6719 [00:00<?, ? examples/s]

train [hf] total= 19654  closed= 9751  open= 9903
test  [hf] total=  6719  closed= 3362  open= 3357


## 3. 모델 로드 — Phi-3.5-Vision + Phi3.5V-Med LoRA

- 로컬 `Phi-3.5-vision-instruct/` 에는 repo 의 `processing_phi3_v_cfg.py` / `processor_config.json` 만 있으므로,
  HF `microsoft/Phi-3.5-vision-instruct` 파일을 **병합**해야 합니다 (repo 의 `processor_config.json` 은 덮어쓰지 말 것).
- 이 커스텀 프로세서는 `qs_highlighted_parts=None` 이면 `txt_highlight_mask` 에서 `None` 을 순회하다 에러가 나고,
  출력에 항상 `highlight_attention_mask` 가 포함됩니다 → 베이스라인에서는 `[]` 를 넘기고 해당 키를 제거합니다.

In [7]:
# ── Phi-3.5-vision-instruct HF 가중치 자동 다운로드 ────────────────────────
# repo 의 processor_config.json / processing_phi3_v_cfg.py 를 덮어쓰지 않고
# microsoft/Phi-3.5-vision-instruct 의 나머지 파일만 병합합니다.
import shutil
from huggingface_hub import snapshot_download

_SKIP_FILES = {'processor_config.json', 'processing_phi3_v_cfg.py'}

if not os.path.exists(os.path.join(BASE_MODEL_DIR, 'config.json')):
    print('Phi-3.5-vision-instruct 가중치 다운로드 중 (약 8-9 GB)...')
    _hf_tmp = snapshot_download(
        'microsoft/Phi-3.5-vision-instruct',
        local_dir='/tmp/phi35_hf',
        ignore_patterns=['*.git', '__pycache__'],
    )
    os.makedirs(BASE_MODEL_DIR, exist_ok=True)
    for _fname in os.listdir(_hf_tmp):
        if _fname in _SKIP_FILES:
            continue
        _src = os.path.join(_hf_tmp, _fname)
        _dst = os.path.join(BASE_MODEL_DIR, _fname)
        if os.path.isdir(_src):
            shutil.copytree(_src, _dst, dirs_exist_ok=True)
        else:
            shutil.copy2(_src, _dst)
    print('다운로드 완료. 파일 목록:', sorted(os.listdir(BASE_MODEL_DIR)))
else:
    print('가중치 이미 존재:', os.path.join(BASE_MODEL_DIR, 'config.json'))

Phi-3.5-vision-instruct 가중치 다운로드 중 (약 8-9 GB)...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 21 files:   0%|          | 0/21 [00:00<?, ?it/s]

다운로드 완료. 파일 목록: ['.cache', '.gitattributes', 'CODE_OF_CONDUCT.md', 'LICENSE', 'README.md', 'SECURITY.md', 'SUPPORT.md', 'config.json', 'configuration_phi3_v.py', 'data_summary_card.md', 'generation_config.json', 'model-00001-of-00002.safetensors', 'model-00002-of-00002.safetensors', 'model.safetensors.index.json', 'modeling_phi3_v.py', 'preprocessor_config.json', 'processing_phi3_v.py', 'processing_phi3_v_cfg.py', 'processor_config.json', 'sample_inference.py', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json']


In [10]:
from transformers import AutoModelForCausalLM, AutoProcessor
from peft import PeftModel, LoraConfig, get_peft_model

assert os.path.exists(os.path.join(BASE_MODEL_DIR, 'config.json')), \
    f'{BASE_MODEL_DIR} 에 HF 가중치(config.json, *.safetensors 등)를 병합하세요'
with open(os.path.join(BASE_MODEL_DIR, 'processor_config.json')) as f:
    assert 'processing_phi3_v_cfg' in f.read(), 'processor_config.json 이 HF 원본으로 덮어써졌습니다 — repo 버전으로 복원하세요'

processor = AutoProcessor.from_pretrained(BASE_MODEL_DIR, trust_remote_code=True, num_crops=NUM_CROPS)

def resolve_med_lora():
    if os.path.exists(os.path.join(MED_LORA_DIR, 'adapter_config.json')):
        return MED_LORA_DIR
    from huggingface_hub import snapshot_download
    local = snapshot_download(MED_LORA_HF, local_dir='./lora_weights/Phi-3.5V-Med')
    found = glob.glob(os.path.join(local, '**', 'adapter_config.json'), recursive=True)
    assert found, f'{MED_LORA_HF} 에서 adapter_config.json 을 찾지 못했습니다'
    return os.path.dirname(found[0])

med_lora_path = resolve_med_lora()
print('Med LoRA :', med_lora_path)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_DIR,
    torch_dtype=torch.bfloat16,          # §4.3 bfloat16
    trust_remote_code=True,
    _attn_implementation='eager',        # flash_attn 설치 시 'flash_attention_2' 로 변경 가능
)

if DOWNSTREAM_MODE == 'merge_then_new':
    model = PeftModel.from_pretrained(model, med_lora_path).merge_and_unload()
    model = get_peft_model(model, LoraConfig(
        r=LORA_RANK,
        lora_alpha=LORA_RANK * LORA_ALPHA_RATIO,
        lora_dropout=LORA_DROPOUT,
        target_modules=LORA_TARGET,
        init_lora_weights=True,          # 신버전 PEFT: 'gaussian' 대신 True
        task_type='CAUSAL_LM',
    ))
elif DOWNSTREAM_MODE == 'continue':
    model = PeftModel.from_pretrained(model, med_lora_path, is_trainable=True)
else:
    raise ValueError(DOWNSTREAM_MODE)

if FREEZE_VISION:
    for n, p in model.named_parameters():
        if 'vision_embed_tokens' in n:
            p.requires_grad_(False)

model.print_trainable_parameters()
model.to(DEVICE)

[transformers] Missing validation function in 'RotaryEmbeddingConfigMixin' for 'rope_type'='su'


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

[transformers] Missing validation function in 'RotaryEmbeddingConfigMixin' for 'rope_type'='su'
[transformers] Phi3VForCausalLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly defined. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.


Med LoRA : /content/Expert-CFG/lora_weights/Phi-3.5V-Med


Loading weights:   0%|          | 0/592 [00:00<?, ?it/s]

[transformers] Phi3VForCausalLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly defined. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.


trainable params: 100,663,296 || all params: 4,247,284,736 || trainable%: 2.3701


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Phi3VForCausalLM(
      (model): Phi3VModel(
        (embed_tokens): Embedding(32064, 3072, padding_idx=32000)
        (embed_dropout): Dropout(p=0.0, inplace=False)
        (vision_embed_tokens): Phi3ImageEmbedding(
          (drop): Dropout(p=0.0, inplace=False)
          (wte): Embedding(32064, 3072, padding_idx=32000)
          (img_processor): CLIPVisionModel(
            (embeddings): CLIPVisionEmbeddings(
              (patch_embedding): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14), bias=False)
              (position_embedding): Embedding(577, 1024)
            )
            (pre_layrnorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
            (encoder): CLIPEncoder(
              (layers): ModuleList(
                (0-23): 24 x CLIPEncoderLayer(
                  (self_attn): CLIPAttention(
                    (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
                  

## 4. Downstream fine-tuning (PathVQA train)

§4.3: AdamW, cosine, lr 1e-4, batch 64, 10 epochs, bf16. Phi-3-V 는 batch 1 만 지원하므로 grad accumulation 으로 64 를 맞춥니다.
프롬프트/라벨 구성은 `src/datasets/pathvqa.py` 의 `PathVQADataCollator` 와 동일 (답변 토큰에만 loss).

> 참고: train ≈ 19.7k 문항 × 5 epoch, batch 1 → 단일 3090 기준 수 일 소요될 수 있습니다.

In [14]:
from transformers import Trainer, TrainingArguments
import inspect

class PathVQATrainCollator:
    def __init__(self, processor):
        self.processor = processor

    def __call__(self, examples):
        assert len(examples) == 1, 'Phi-3-V only supports batch_size == 1'
        ex = examples[0]
        prompt = self.processor.tokenizer.apply_chat_template(
            [{'role': 'user', 'content': f"<|image_1|>\n{ex['question']}"}],
            tokenize=False, add_generation_prompt=True)
        answer = f"{ex['answer']}<|end|>\n<|endoftext|>"

        batch = self.processor(prompt, [ex['image']], return_tensors='pt', qs_highlighted_parts=[])
        del batch['highlight_attention_mask']
        prompt_ids = batch['input_ids']
        answer_ids = self.processor.tokenizer(answer, add_special_tokens=False, return_tensors='pt')['input_ids']

        batch['input_ids'] = torch.cat([prompt_ids, answer_ids], dim=1)
        batch['labels'] = torch.cat([torch.full_like(prompt_ids, -100), answer_ids], dim=1)
        del batch['attention_mask']
        return batch

_ta_params = set(inspect.signature(TrainingArguments.__init__).parameters)

# warmup 스텝 수 계산 (warmup_ratio 가 없을 때 warmup_steps 로 대체)
_total_steps = (len(train_ds) * EPOCHS + GRAD_ACCUM - 1) // GRAD_ACCUM
_warmup_steps_val = int(_total_steps * WARMUP_RATIO)

_base_kwargs = dict(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BS,
    gradient_accumulation_steps=GRAD_ACCUM,
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    logging_steps=10,
    save_strategy='epoch',
    save_total_limit=2,
    save_only_model=True,
    remove_unused_columns=False,
    dataloader_num_workers=0,
    report_to='none',
    seed=SEED,
)

# 버전별 파라미터 호환: warmup_ratio(4.x) → warmup_steps(5.x fallback)
_v4_kwargs = dict(
    learning_rate=LR,
    optim='adamw_torch',
    lr_scheduler_type='cosine',
    weight_decay=WEIGHT_DECAY,
)
if 'warmup_ratio' in _ta_params:
    _v4_kwargs['warmup_ratio'] = WARMUP_RATIO
elif 'warmup_steps' in _ta_params:
    _v4_kwargs['warmup_steps'] = _warmup_steps_val
    print(f'warmup_ratio → warmup_steps={_warmup_steps_val} ({WARMUP_RATIO*100:.1f}% of {_total_steps} steps)')

_compat = {k: v for k, v in _v4_kwargs.items() if k in _ta_params}
_dropped = [k for k in _v4_kwargs if k not in _ta_params]
if _dropped:
    print(f'[경고] 무시된 파라미터: {_dropped}')

training_args = TrainingArguments(**_base_kwargs, **_compat)

trainer = Trainer(model=model, args=training_args,
                  train_dataset=train_ds, data_collator=PathVQATrainCollator(processor))

# 체크포인트가 있으면 이어받고, 없으면 처음부터 시작
_ckpt_dirs = sorted(glob.glob(os.path.join(OUTPUT_DIR, 'checkpoint-*')))
_resume = _ckpt_dirs[-1] if _ckpt_dirs else False
print(f'resume_from_checkpoint = {_resume}')

trainer.train(resume_from_checkpoint=_resume)
trainer.save_model(OUTPUT_DIR)

warmup_ratio → warmup_steps=46 (3.0% of 1536 steps)
resume_from_checkpoint = False


/usr/local/lib/python3.13/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.13/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


OutOfMemoryError: CUDA out of memory. Tried to allocate 462.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 205.81 MiB is free. Including non-PyTorch memory, this process has 14.36 GiB memory in use. Of the allocated memory 13.68 GiB is allocated by PyTorch, and 559.36 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

## 5. 평가 지표 & Predictive entropy

- **Closed**: accuracy — LLaVA-Med `run_eval.py` 처럼 GT(yes/no) 가 예측에 포함되면 정답 (여기서는 단어 단위로 매칭)
- **Open**: recall — GT 토큰 중 예측에 등장한 비율
- **Overall**: 모든 문항 점수의 평균 (문항 수 가중)
- **Entropy (Eq.4)**: $\mathrm{PE} = -\frac{1}{N}\sum_{i}\sum_{x_i} P(x_i|x_{<i})\log P(x_i|x_{<i})$ — greedy 생성 시 각 스텝 분포의 엔트로피 평균

In [ ]:
import re
from collections import Counter

def normalize(text):
    text = text.lower().replace('<|end|>', ' ')
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    return text.split()

def closed_accuracy(pred, gt):
    return float(gt.strip().lower() in normalize(pred))

def open_recall(pred, gt):
    gt_toks, pred_toks = Counter(normalize(gt)), Counter(normalize(pred))
    if not gt_toks:
        return 0.0
    return sum((gt_toks & pred_toks).values()) / sum(gt_toks.values())

def predictive_entropy(step_scores):
    # step_scores: generate(output_scores=True).scores — 스텝별 (1, vocab) logits
    if len(step_scores) == 0:
        return 0.0
    logits = torch.stack([s[0] for s in step_scores]).float()     # (N, vocab)
    logp = torch.log_softmax(logits, dim=-1)
    return (-(logp.exp() * logp).sum(-1)).mean().item()

assert closed_accuracy('Yes', 'yes') == 1 and closed_accuracy('no', 'yes') == 0
assert open_recall('giant cells and fibrosis are seen', 'giant cells and fibrosis') == 1.0

## 6. 테스트셋 평가 (greedy)

In [ ]:
from tqdm.auto import tqdm

@torch.no_grad()
def evaluate(model, processor, dataset, max_samples=None):
    model.eval()
    end_id = processor.tokenizer.convert_tokens_to_ids('<|end|>')
    records = []
    n = len(dataset) if max_samples is None else min(max_samples, len(dataset))
    for i in tqdm(range(n)):
        ex = dataset[i]
        prompt = processor.tokenizer.apply_chat_template(
            [{'role': 'user', 'content': f"<|image_1|>\n{ex['question']}"}],
            tokenize=False, add_generation_prompt=True)
        inputs = processor(prompt, [ex['image']], return_tensors='pt', qs_highlighted_parts=[])
        del inputs['highlight_attention_mask']
        inputs = inputs.to(DEVICE)

        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            num_beams=NUM_BEAMS,
            do_sample=False,
            eos_token_id=[processor.tokenizer.eos_token_id, end_id],
            output_scores=True,
            return_dict_in_generate=True,
        )
        gen_ids = out.sequences[:, inputs['input_ids'].size(1):]
        pred = processor.batch_decode(gen_ids, skip_special_tokens=True,
                                      clean_up_tokenization_spaces=False)[0].strip().strip('.')

        score = (closed_accuracy if ex['answer_type'] == 'CLOSED' else open_recall)(pred, ex['answer'])
        records.append({'idx': i, 'question': ex['question'], 'answer': ex['answer'],
                        'answer_type': ex['answer_type'], 'prediction': pred,
                        'score': score, 'entropy': predictive_entropy(out.scores)})
    return records

def summarize(records):
    closed = [r['score'] for r in records if r['answer_type'] == 'CLOSED']
    opened = [r['score'] for r in records if r['answer_type'] == 'OPEN']
    return {'open': 100 * np.mean(opened) if opened else float('nan'),
            'closed': 100 * np.mean(closed) if closed else float('nan'),
            'overall': 100 * np.mean([r['score'] for r in records]),
            'n_open': len(opened), 'n_closed': len(closed)}

# 빠른 확인은 max_samples=200 등으로
records = evaluate(model, processor, test_ds, max_samples=None)
with open(RESULT_JSON, 'w') as f:
    json.dump(records, f, indent=2, ensure_ascii=False)

m = summarize(records)
paper = {'open': 37.42, 'closed': 92.89, 'overall': 65.24}
print(f"{'':8s}{'paper':>8s}{'ours':>8s}{'diff':>8s}")
for k in ('open', 'closed', 'overall'):
    print(f'{k:8s}{paper[k]:8.2f}{m[k]:8.2f}{m[k]-paper[k]:+8.2f}')
print(f"n_open={m['n_open']}, n_closed={m['n_closed']}")

## 7. 불확실성 추정 검증 (Figure 3: PathVQA AUC = 0.91)

엔트로피로 오답(incorrect)을 판별하는 ROC-AUC. 이후 Expert-CFG 에서 상위 5% 고엔트로피 샘플을 선택할 때 그대로 사용합니다.
open 문항은 recall == 1 일 때 정답으로 간주했습니다 (논문 미명시).

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt

ent = np.array([r['entropy'] for r in records])
wrong = np.array([0 if r['score'] == 1.0 else 1 for r in records])

auc = roc_auc_score(wrong, ent)
fpr, tpr, _ = roc_curve(wrong, ent)
print(f'AUC = {auc:.3f}  (paper: 0.91)')

top5 = np.argsort(-ent)[: int(round(0.05 * len(ent)))]
print(f'top-5% 고엔트로피 {len(top5)}개 중 오답 비율: {wrong[top5].mean():.3f}')

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(fpr, tpr, label=f'PathVQA (AUC = {auc:.2f})'); ax[0].plot([0, 1], [0, 1], 'k--', lw=0.8)
ax[0].set_xlabel('False Positive Rate'); ax[0].set_ylabel('True Positive Rate'); ax[0].legend()
ax[1].boxplot([ent[wrong == 0], ent[wrong == 1]], labels=['correct', 'incorrect'])
ax[1].set_ylabel('Predictive entropy')
plt.tight_layout(); plt.show()

## 8. 저장된 체크포인트로 평가만 다시 하기

학습을 건너뛰고 `OUTPUT_DIR` 의 LoRA 로 평가할 때 사용 (3장 모델 로드 대신 실행).

In [ ]:
def load_for_eval(adapter_dir=OUTPUT_DIR):
    base = AutoModelForCausalLM.from_pretrained(BASE_MODEL_DIR, torch_dtype=torch.bfloat16,
                                                trust_remote_code=True, _attn_implementation='eager')
    if DOWNSTREAM_MODE == 'merge_then_new':
        base = PeftModel.from_pretrained(base, resolve_med_lora()).merge_and_unload()
    return PeftModel.from_pretrained(base, adapter_dir).to(DEVICE).eval()

# model = load_for_eval()
# records = evaluate(model, processor, test_ds); print(summarize(records))